In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# ==========================================
# 步驟 1：建立 18 個字元的標準字典（與正式版完全相同）
# ==========================================
special_tokens = ['<pad>', '<eos>']
arithmetic_chars = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '+', '-', '*', '(', ')', '=']
vocab = special_tokens + arithmetic_chars
char_to_id = {ch: idx for idx, ch in enumerate(vocab)}
id_to_char = {idx: ch for idx, ch in enumerate(vocab)}
vocab_size = len(char_to_id)
pad_id = char_to_id['<pad>']
eos_id = char_to_id['<eos>']

# ==========================================
# 步驟 2：生成三位數訓練集 (Dataset Generation)
# ==========================================
def generate_three_digit_dataset(num_samples=30000):
    random.seed(42)
    operators = ['+', '-', '*']
    data = []

    for _ in range(num_samples):
        # 1. 隨機生成 3 個三位數 (100 ~ 999)
        a = random.randint(100, 999)
        b = random.randint(100, 999)
        c = random.randint(100, 999)

        # 2. 隨機選取運算子
        op1 = random.choice(operators)
        op2 = random.choice(operators)

        # 3. 隨機套用 3 種括號結構
        pattern = random.choice([1, 2, 3])
        if pattern == 1:
            formula = f"{a}{op1}{b}{op2}{c}"       # A op1 B op2 C=
        elif pattern == 2:
            formula = f"({a}{op1}{b}){op2}{c}"     # (A op1 B) op2 C=
        else:
            formula = f"{a}{op1}({b}{op2}{c})"     # A op1 (B op2 C)=

        try:
            answer = eval(formula)
            src = formula + "="
            tgt = str(answer)
            data.append({'src': src, 'tgt': tgt})
        except ZeroDivisionError:
            continue

    return pd.DataFrame(data)

print("正在生成 30,000 筆三位數資料集...")
df_train_3digit = generate_three_digit_dataset(30000)
print("生成完畢！範例：")
print(df_train_3digit.head(3))

# ==========================================
# 步驟 3：資料前處理與 DataLoader
# ==========================================
def preprocess_data(df):
    srcs = df['src'].values
    tgts = df['tgt'].astype(str).values
    char_id_list = []
    label_id_list = []

    for s, t in zip(srcs, tgts):
        full_str = s + t
        c_ids = [char_to_id[c] for c in full_str] + [eos_id]
        prefix_len = len(s)
        l_ids = [pad_id] * prefix_len + [char_to_id[c] for c in t] + [eos_id]
        char_id_list.append(c_ids)
        label_id_list.append(l_ids)

    df['char_id_list'] = char_id_list
    df['label_id_list'] = label_id_list
    return df

df_train_3digit = preprocess_data(df_train_3digit)

class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, index):
        x = self.sequences.iloc[index]['char_id_list']
        y = self.sequences.iloc[index]['label_id_list']
        return x, y

def collate_fn(batch):
    batch_x = [torch.tensor(x) for x, y in batch]
    batch_y = [torch.tensor(y) for x, y in batch]
    batch_x_lens = [len(x) for x in batch_x]
    batch_y_lens = [len(y) for y in batch_y]
    batch_x = torch.nn.utils.rnn.pad_sequence(batch_x, batch_first=True, padding_value=pad_id)
    batch_y = torch.nn.utils.rnn.pad_sequence(batch_y, batch_first=True, padding_value=pad_id)
    return batch_x, batch_y, batch_x_lens, batch_y_lens

ds_train_3digit = Dataset(df_train_3digit[['char_id_list', 'label_id_list']])
dl_train_3digit = torch.utils.data.DataLoader(ds_train_3digit, batch_size=128, shuffle=True, collate_fn=collate_fn)

# ==========================================
# 步驟 4：模型架構（使用您調好的 3-layer LSTM）
# ==========================================
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512):
        super(CharRNN, self).__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = torch.nn.LSTM(embed_dim, hidden_dim, num_layers=3, dropout=0.1, batch_first=True)
        self.linear = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, vocab_size)
        )

    def forward(self, batch_x, batch_x_lens):
        x = self.embedding(batch_x)
        x = torch.nn.utils.rnn.pack_padded_sequence(x, batch_x_lens, batch_first=True, enforce_sorted=False)
        x, _ = self.lstm(x)
        x, _ = torch.nn.utils.rnn.pad_packed_sequence(x, batch_first=True)
        return self.linear(x)

    def generator(self, start_char, max_len=200):
        char_list = [char_to_id[c] for c in start_char]
        while len(char_list) < max_len:
            input_tensor = torch.tensor([char_list], dtype=torch.long, device=next(self.parameters()).device)
            out = self.embedding(input_tensor)
            out, _ = self.lstm(out)
            out = self.linear(out)
            y = out[0, -1, :]
            next_char = torch.argmax(y).item()
            if next_char == eos_id:
                break
            char_list.append(next_char)
        return [id_to_char[ch_id] for ch_id in char_list]

model_3digit = CharRNN(vocab_size).to(device)
criterion = torch.nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.AdamW(model_3digit.parameters(), lr=0.001, weight_decay=1e-4)

# ==========================================
# 步驟 5：在三位數資料集上訓練 5 輪
# ==========================================
print("\n開始訓練三位數模型 (5 Epochs)...")
model_3digit.train()
for epoch in range(1, 6):
    bar = tqdm(dl_train_3digit, desc=f"Train 3-digit Epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        optimizer.zero_grad()
        pred = model_3digit(batch_x.to(device), batch_x_lens)
        loss = criterion(pred.view(-1, vocab_size), batch_y.to(device).view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_value_(model_3digit.parameters(), 1.0)
        optimizer.step()
        bar.set_postfix(loss=loss.item())

# ==========================================
# 步驟 6：拿官方兩位數評估集 (arithmetic_eval.csv) 進行測試！
# ==========================================
print("\n正在讀取官方兩位數評估集 (arithmetic_eval.csv)...")
# 請確認您的雲端或本地資料路徑
eval_path = '/content/drive/MyDrive/nlp-hw2/arithmetic_eval.csv'  # 若在不同路徑請修改
df_eval = pd.read_csv(eval_path)

model_3digit.eval()
matched = 0
total = 0
examples = []

print("正在評估三位數模型在兩位數考卷上的表現 (已開啟加速)...")
# 評估官方 10,000 筆兩位數評估集
for idx, row in tqdm(df_eval.iterrows(), total=len(df_eval)):
    src = row['src']
    tgt = str(row['tgt'])

    # 🌟 關鍵加速：限制最多只生成算式長度 + 10 個字元，不用傻等 200 字
    pred_tokens = model_3digit.generator(src, max_len=len(src) + 10)
    pred_ans = ''.join(pred_tokens[len(src):])

    if pred_ans == tgt:
        matched += 1
    total += 1

    # 挑選前 5 筆錯誤案例留存
    if len(examples) < 5 and pred_ans != tgt:
        examples.append((src, tgt, pred_ans))

em_acc = matched / total
print("\n==========================================")
print(f"🔥 三位數模型在兩位數評估集上的 EM 準確率: {em_acc:.4f} ({em_acc*100:.2f}%)")
print("==========================================")
print("\n❌ 錯誤預測範例對照（輸入 ➔ 真實答案 ➔ 模型預測）：")
for src, tgt, pred in examples:
    print(f"算式: {src:<15} | 真實答案: {tgt:<8} | 模型預測: {pred}")

Using device: cuda
正在生成 30,000 筆三位數資料集...
生成完畢！範例：
              src     tgt
0    754*214-125=  161231
1  328+(242*854)=  206996
2    658-189+704=    1173

開始訓練三位數模型 (5 Epochs)...


Train 3-digit Epoch 5: 100%|██████████| 235/235 [00:04<00:00, 48.87it/s, loss=4.27e-7]



正在讀取官方兩位數評估集 (arithmetic_eval.csv)...
正在評估三位數模型在兩位數考卷上的表現 (已開啟加速)...


100%|██████████| 10000/10000 [02:11<00:00, 76.13it/s]


🔥 三位數模型在兩位數評估集上的 EM 準確率: 0.0000 (0.00%)

❌ 錯誤預測範例對照（輸入 ➔ 真實答案 ➔ 模型預測）：
算式: 34+47-32=       | 真實答案: 49       | 模型預測: 7777777777
算式: 33-41-15=       | 真實答案: -23      | 模型預測: 7777777777
算式: 31-(36*44)=     | 真實答案: -1553    | 模型預測: 7777777777
算式: 21*(41+3)=      | 真實答案: 924      | 模型預測: 7777777777
算式: (47-27)+26=     | 真實答案: 46       | 模型預測: 7777777777
